# FireNet Training — FRED Dataset

Fine-tune **FireNet** (GRU recurrent UNet, 38 k params) from pretrained weights using FRED event-camera data.

| Setting | Value |
|---|---|
| Model | FireNet from `pretrained/firenet_1000.pth.tar`, structure unchanged |
| Input | Voxel grid (5 bins, 256×256), events from `Event/events.npy` |
| GT | Grayscale of RGB frame, matched by relative timestamp |
| Loss | VGG16 perceptual (proxy for LPIPS) + temporal consistency L1 |
| Optim | Adam lr=1e-4, 1000 epochs, sequence length L=20, λ_TC=2 |


In [ ]:
import sys, os
from pathlib import Path
import torch

# Notebook lives in reconstruct/firenet/train/
NOTEBOOK_DIR = Path(os.getcwd())
FIRENET_DIR  = NOTEBOOK_DIR.parent       # reconstruct/firenet/
REPO         = FIRENET_DIR.parents[1]    # AMI-KK/

# model/ and base/ packages live inside FIRENET_DIR
if str(FIRENET_DIR) not in sys.path:
    sys.path.insert(0, str(FIRENET_DIR))
os.chdir(FIRENET_DIR)   # 'from base import BaseModel' requires cwd = firenet/

DATA_DIR   = REPO / 'data/FRED/train'
PRETRAINED = FIRENET_DIR / 'pretrained/firenet_1000.pth.tar'
CACHE_DIR  = REPO / 'cache/firenet_fred'
SAVE_DIR   = REPO / 'checkpoints/firenet'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE   = (256, 256)   # (H_out, W_out) spatial resize for voxels and GT
NUM_BINS   = 5
SEQ_LEN    = 20           # L in the paper: consecutive frames per training sequence
L0         = 10           # temporal consistency loss starts from this step index
LAMBDA_TC  = 2.0
EPOCHS     = 1000
LR         = 1e-4
BATCH_SIZE = 2
VAL_FRAC   = 0.1

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


In [ ]:
import numpy as np
import cv2


def parse_rgb_ts(stem):
    """Absolute seconds from midnight. Filename: Video_0_16_03_03.363444"""
    p = stem.split('_')
    return int(p[-3]) * 3600 + int(p[-2]) * 60 + float(p[-1])


def events_to_voxel(ev, t0, t1, H, W, num_bins, img_size):
    """Build (num_bins, H_out, W_out) voxel for events with t in [t0, t1]."""
    t  = ev['t']
    i0 = int(np.searchsorted(t, t0, 'left'))
    i1 = int(np.searchsorted(t, t1, 'right'))
    H_out, W_out = img_size

    if i1 - i0 < 2:
        return np.zeros((num_bins, H_out, W_out), dtype=np.float32)

    x  = ev['x'][i0:i1].astype(np.int64)
    y  = ev['y'][i0:i1].astype(np.int64)
    p  = np.where(ev['p'][i0:i1] > 0, 1.0, -1.0).astype(np.float32)
    ts = t[i0:i1]

    t_norm = (ts - ts[0]) / max(ts[-1] - ts[0], 1e-9) * (num_bins - 1)
    tb     = np.clip(t_norm.astype(np.int64), 0, num_bins - 2)
    alpha  = (t_norm - tb).astype(np.float32)

    valid = (x >= 0) & (x < W) & (y >= 0) & (y < H)
    x, y, tb, alpha, p = x[valid], y[valid], tb[valid], alpha[valid], p[valid]

    HW   = H * W
    flat = tb * HW + y * W + x
    vox  = (
        np.bincount(flat,      weights=(1 - alpha) * p, minlength=num_bins * HW) +
        np.bincount(flat + HW, weights=alpha * p,       minlength=num_bins * HW)
    ).reshape(num_bins, H, W).astype(np.float32)

    for b in range(num_bins):
        m = np.abs(vox[b]).max()
        if m > 0:
            vox[b] /= m

    return np.stack([
        cv2.resize(vox[b], (W_out, H_out), interpolation=cv2.INTER_LINEAR)
        for b in range(num_bins)
    ])


In [ ]:
# Precompute voxels and grayscale GTs for all sequences and cache to disk.
# Safe to re-run: skips sequences whose cache files already exist.

def precompute_sequence(seq_dir):
    name     = seq_dir.name
    vox_path = CACHE_DIR / f'{name}_voxels.npy'
    gt_path  = CACHE_DIR / f'{name}_gts.npy'

    if vox_path.exists() and gt_path.exists():
        print(f'  {name}: cache exists, skip')
        return

    ev_raw = np.load(str(seq_dir / 'Event/events.npy'))
    t_ev   = ev_raw['t'].astype(np.float64) / 1e6   # microsec -> sec
    t_ev  -= t_ev[0]                                  # relative from first event
    ev = {'x': ev_raw['x'], 'y': ev_raw['y'], 'p': ev_raw['p'], 't': t_ev}
    H  = int(ev_raw['y'].max()) + 1
    W  = int(ev_raw['x'].max()) + 1

    rgb_files = sorted(
        (seq_dir / 'RGB').glob('*.jpg'),
        key=lambda f: parse_rgb_ts(f.stem)
    )
    rgb_ts  = np.array([parse_rgb_ts(f.stem) for f in rgb_files])
    rgb_ts -= rgb_ts[0]   # relative from first RGB frame; aligns with t_ev
    N       = len(rgb_files)
    H_out, W_out = IMG_SIZE

    voxels = np.zeros((N - 1, NUM_BINS, H_out, W_out), dtype=np.float32)
    gts    = np.zeros((N,     1,        H_out, W_out), dtype=np.float32)

    for k in range(N):
        if k % 500 == 0:
            print(f'  {name}: {k}/{N}', end='\r', flush=True)
        img  = cv2.imread(str(rgb_files[k]))
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
        gts[k, 0] = cv2.resize(gray, (W_out, H_out), interpolation=cv2.INTER_LINEAR)
        if k > 0:
            # voxels[k-1]: events from rgb_ts[k-1] to rgb_ts[k]; GT = gts[k]
            voxels[k - 1] = events_to_voxel(
                ev, rgb_ts[k - 1], rgb_ts[k], H, W, NUM_BINS, IMG_SIZE
            )

    np.save(str(vox_path), voxels)
    np.save(str(gt_path),  gts)
    print(f'  {name}: saved {N - 1} voxels + {N} GTs       ')


for seq_dir in sorted(DATA_DIR.iterdir()):
    if seq_dir.is_dir():
        precompute_sequence(seq_dir)
print('Precompute done.')


In [ ]:
import random
import torch
from torch.utils.data import Dataset, DataLoader, Subset


class FREDDataset(Dataset):
    """Sequences of (voxel_grid, grayscale_GT) pairs loaded from precomputed cache."""

    def __init__(self, augment=False):
        self.augment = augment
        self.samples = []   # list of (voxels_mmap, gts_mmap, start_idx)

        for seq_dir in sorted(DATA_DIR.iterdir()):
            if not seq_dir.is_dir():
                continue
            vox_path = CACHE_DIR / f'{seq_dir.name}_voxels.npy'
            gt_path  = CACHE_DIR / f'{seq_dir.name}_gts.npy'
            if not vox_path.exists():
                continue

            # mmap_mode='r' lets OS page in slices on demand, keeps RAM low
            voxels = np.load(str(vox_path), mmap_mode='r')   # (N-1, bins, H, W)
            gts    = np.load(str(gt_path),  mmap_mode='r')   # (N,   1,    H, W)
            N = voxels.shape[0]

            for start in range(0, N - SEQ_LEN + 1, SEQ_LEN // 2):
                self.samples.append((voxels, gts, start))

        print(f'Dataset: {len(self.samples)} sequences (augment={augment})')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        voxels, gts, start = self.samples[idx]
        v = voxels[start     : start + SEQ_LEN    ].copy()   # (L, bins, H, W)
        g = gts   [start + 1 : start + SEQ_LEN + 1].copy()   # (L, 1,    H, W)
        if self.augment and random.random() < 0.5:
            v = v[..., ::-1].copy()   # horizontal flip
            g = g[..., ::-1].copy()
        return torch.from_numpy(v), torch.from_numpy(g)


# Two instances so train gets augmentation, val does not
ds_aug   = FREDDataset(augment=True)
ds_noaug = FREDDataset(augment=False)
n        = len(ds_aug)
n_val    = max(1, int(n * VAL_FRAC))
n_train  = n - n_val
idx      = list(range(n))

train_ds = Subset(ds_aug,   idx[:n_train])
val_ds   = Subset(ds_noaug, idx[n_train:])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
print(f'Train: {n_train} / Val: {n_val} | Batches: {len(train_loader)} / {len(val_loader)}')


In [ ]:
import torch.nn as nn
from model.model import FireNet

ckpt   = torch.load(str(PRETRAINED), map_location='cpu', weights_only=False)
config = ckpt['config']['model']   # num_bins=5, recurrent_block_type='convgru', etc.

model = FireNet(config).to(DEVICE)
model.load_state_dict(ckpt['state_dict'])
model.train()

n_p = sum(p.numel() for p in model.parameters())
print(f'FireNet loaded: {n_p:,} params  (pretrained from epoch {ckpt["epoch"]})')


In [ ]:
import torch.nn.functional as F
import torchvision.models as tv_models


class PerceptualLoss(nn.Module):
    """
    VGG16 feature-based perceptual loss used as a proxy for LPIPS.
    Computes L1 between relu1_2, relu2_2, relu3_3 feature maps.
    VGG weights are frozen; gradients flow to model predictions only.
    """

    def __init__(self):
        super().__init__()
        feats    = list(tv_models.vgg16(weights='DEFAULT').features)
        self.s1  = nn.Sequential(*feats[:4])    # relu1_2
        self.s2  = nn.Sequential(*feats[4:9])   # relu2_2
        self.s3  = nn.Sequential(*feats[9:16])  # relu3_3
        for p in self.parameters():
            p.requires_grad_(False)
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std',  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, pred, target):
        # pred, target: (B, 1, H, W) grayscale in [0, 1]
        p = (pred.repeat(1, 3, 1, 1)   - self.mean) / self.std
        t = (target.repeat(1, 3, 1, 1) - self.mean) / self.std
        loss = 0.0
        for sl in (self.s1, self.s2, self.s3):
            p = sl(p)
            t = sl(t)
            loss = loss + F.l1_loss(p, t)
        return loss


perc_loss = PerceptualLoss().to(DEVICE)
print('PerceptualLoss (VGG16) ready')


In [ ]:
import time

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scaler    = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))


def run_epoch(loader, train):
    model.train(train)
    total, n = 0.0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for voxels, gts in loader:
            voxels = voxels.to(DEVICE, non_blocking=True)   # (B, L, bins, H, W)
            gts    = gts.to(DEVICE,    non_blocking=True)   # (B, L, 1,    H, W)

            with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == 'cuda')):
                states = None
                preds  = []
                for k in range(SEQ_LEN):
                    img, states = model(voxels[:, k], states)
                    preds.append(torch.sigmoid(img))   # model has no output activation

                # Reconstruction loss: VGG perceptual on all L frames
                loss_r = sum(perc_loss(preds[k], gts[:, k]) for k in range(SEQ_LEN))

                # Temporal consistency: L1 between consecutive frames from L0 onward
                # Simplification: direct L1 instead of flow-warped photometric error
                loss_tc = sum(
                    F.l1_loss(preds[k - 1], preds[k])
                    for k in range(L0, SEQ_LEN)
                )

                loss = loss_r + LAMBDA_TC * loss_tc

            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            total += loss.item()
            n     += 1

    return total / max(n, 1)


best_val = float('inf')
history  = []

for epoch in range(1, EPOCHS + 1):
    t0      = time.time()
    t_loss  = run_epoch(train_loader, train=True)
    v_loss  = run_epoch(val_loader,   train=False)
    elapsed = time.time() - t0
    history.append((t_loss, v_loss))

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:4d}/{EPOCHS}  {elapsed:4.0f}s  train={t_loss:.4f}  val={v_loss:.4f}')

    ckpt_state = {
        'epoch':     epoch,
        'model':     model.state_dict(),
        'optimizer': optimizer.state_dict(),
    }
    torch.save(ckpt_state, str(SAVE_DIR / 'last.pth'))
    if v_loss < best_val:
        best_val = v_loss
        torch.save(ckpt_state, str(SAVE_DIR / 'best.pth'))
        print(f'  best val -> {best_val:.4f}')

print(f'\nDone. Best val: {best_val:.4f}')


In [ ]:
import matplotlib.pyplot as plt

train_l, val_l = zip(*history)
plt.figure(figsize=(9, 3))
plt.plot(train_l, label='train')
plt.plot(val_l,   label='val')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(str(SAVE_DIR / 'loss.png'), dpi=120)
plt.show()

# Sample reconstructions from validation set
model.eval()
voxels, gts = next(iter(val_loader))
voxels = voxels.to(DEVICE)
states = None
preds  = []
with torch.no_grad():
    for k in range(SEQ_LEN):
        img, states = model(voxels[:, k], states)
        preds.append(torch.sigmoid(img)[0, 0].cpu().numpy())

show = [0, 4, 9, 14, 19]
fig, axes = plt.subplots(2, len(show), figsize=(3 * len(show), 5))
for col, k in enumerate(show):
    axes[0, col].imshow(gts[0, k, 0].numpy(), cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(f't={k}')
    axes[0, col].axis('off')
    axes[1, col].imshow(preds[k], cmap='gray', vmin=0, vmax=1)
    axes[1, col].axis('off')
axes[0, 0].set_ylabel('GT')
axes[1, 0].set_ylabel('Pred')
plt.tight_layout()
plt.savefig(str(SAVE_DIR / 'samples.png'), dpi=120)
plt.show()
